# Phase 9b — what controls the routing discontinuity at the switch?

Phase 9 established a small but significant, position-matched routing discontinuity
at the language-switch token (+0.037 [+0.028,+0.046]). This notebook asks **which
parameters of the measurement significantly influence it**, and looks at the
phenomenon from several perspectives.

**Design.** The teacher-forced forward is *deterministic* — rerunning the identical
transcript reproduces the identical routing. So the right "multiple runs" are not
stochastic repeats but **parameter sweeps**: one capture pass stores the **raw
routing tensors** `R [layers, seq, top_k]` per transcript on Drive (~0.5 MB each),
and every analysis below is then a pure offline re-analysis — no further GPU time.

Perspectives:
1. **Window `W`** (2→64): is the discontinuity a short-range or long-range routing break?
2. **Layer-resolved**: *where* in the 40 layers does it live — early (lexical/script) or late (semantic)?
3. **Top-k depth** (1→8): does the *dominant* expert change, or only the tail of the top-8?
4. **Control behaviour** (two different fluent controls): does the effect depend on the control choice?
5. **Entry vs sustained**: onset token vs later foreign-script tokens — separates *entering the mode*
   from *being in a foreign script* (the script confound).

Cells 1–6 need the A100 (capture, ~one forward per transcript, resumable).
Cells 7+ need only numpy + the Drive captures — they run in any session.


In [ ]:
# === Cell 1 — config =========================================================
import os
try:
    from google.colab import userdata
    v=None
    try: v=userdata.get("HF_TOKEN")
    except Exception: v=None
    if v: os.environ.setdefault("HF_TOKEN",v)
except Exception as e:
    print("colab secrets unavailable:", e)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")

MODEL   = os.environ.get("WEIRDSPEC_TARGET_MODEL","Qwen/Qwen3.6-35B-A3B-FP8")
DATA_DIR    = "/content/drive/MyDrive/weirdspec"
CAPTURE_DIR = "/content/drive/MyDrive/weirdspec/routing_caps"   # raw R tensors land here
TIPPED_BEHAVIORS  = ["language-switching-english"]
CONTROL_BEHAVIORS = ["chemtrails-assertion", "recommends-drunk-driving"]  # two controls on purpose
N_PER_CLASS = 60          # per behaviour
MAX_LEN     = 2048
FOREIGN_RUN = 3
MOUNT_DRIVE = True
print("model:", MODEL, "| captures ->", CAPTURE_DIR)


In [ ]:
# === Cell 2 — mount + deps ===================================================
import os, sys, subprocess
if MOUNT_DRIVE:
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e: print("drive mount skipped:", e)
subprocess.run([sys.executable,"-m","pip","install","-q","transformers==5.10.2","accelerate"], check=True)
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# === Cell 3 — load model (same proven path as phase 9) ======================
import torch
from transformers import AutoConfig, AutoModel, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)
cfg = AutoConfig.from_pretrained(MODEL)
tc = getattr(cfg,"text_config",cfg)
def cfgget(o,*names):
    for n in names:
        v=getattr(o,n,None)
        if v is not None: return v
NUM_EXPERTS=cfgget(tc,"num_experts","n_routed_experts","num_local_experts")
TOP_K      =cfgget(tc,"num_experts_per_tok","num_experts_per_token","top_k")
HIDDEN     =cfgget(tc,"hidden_size"); N_LAYERS=cfgget(tc,"num_hidden_layers")
print(f"MoE: num_experts={NUM_EXPERTS} top_k={TOP_K} hidden={HIDDEN} layers={N_LAYERS}")
kwargs={"attn_implementation":"sdpa","device_map":"auto"}
kwargs["dtype"]="auto" if getattr(cfg,"quantization_config",None) is not None else torch.bfloat16
try:
    model=AutoModel.from_pretrained(MODEL, **kwargs)
except ValueError:
    import transformers as tf
    model=getattr(tf,str(cfg.architectures[0])).from_pretrained(MODEL, **kwargs)
model.eval(); DEV=next(model.parameters()).device
print("loaded on", DEV)


In [ ]:
# === Cell 4 — gates + hooks (shape-matched, class-agnostic; proven) =========
import re, numpy as np, torch
def _wshape(mod):
    w=getattr(mod,"weight",None)
    return tuple(w.shape) if (w is not None and hasattr(w,"shape") and w.dim()==2) else None
def is_gate(name, shp, num_experts, hidden):
    if shp not in [(num_experts,hidden),(hidden,num_experts)]: return False
    low=name.lower()
    if any(b in low for b in ("shared","attn","proj")): return False
    return ("gate" in low or "router" in low or "moe" in low)
gates=[]
for name,mod in model.named_modules():
    if is_gate(name,_wshape(mod),NUM_EXPERTS,HIDDEN):
        m=re.search(r"layers\.(\d+)\.",name)
        gates.append((int(m.group(1)) if m else -1,name,mod))
gates.sort(key=lambda g:g[0])
assert gates, "no gates found (see phase 9 Cell 4 dump approach)"
print(f"{len(gates)} gates, e.g. {gates[0][1]}")
_cap={}
def _mkhook(pos):
    def hook(module,inp,out):
        logits=out[0] if isinstance(out,(tuple,list)) else out
        _cap[pos]=torch.topk(logits,TOP_K,dim=-1).indices.detach().to("cpu")
    return hook
for i,g in enumerate(gates): g[2].register_forward_hook(_mkhook(i))
@torch.no_grad()
def route(input_ids):
    _cap.clear()
    ids=torch.tensor([input_ids],device=DEV)
    model(input_ids=ids,use_cache=False)
    seq=ids.shape[1]
    R=np.empty((len(gates),seq,TOP_K),dtype=np.int16)
    for i in range(len(gates)):
        R[i]=_cap[i].numpy().reshape(-1,TOP_K)[:seq].astype(np.int16)
    return R
print("router capture ready")


In [ ]:
# === Cell 5 — transcript helpers (from phase 9) =============================
import os, glob, json, hashlib, numpy as np
def read_jsonl(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def find_file(d,*names):
    for nm in names:
        p=os.path.join(d,nm)
        if os.path.isfile(p): return p
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None
FOREIGN=[(0x0370,0x03FF),(0x0400,0x04FF),(0x0500,0x052F),(0x0530,0x058F),(0x0590,0x05FF),
         (0x0600,0x06FF),(0x0700,0x074F),(0x0900,0x097F),(0x0E00,0x0E7F),
         (0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def is_foreign(ch):
    if not ch.isalpha(): return False
    o=ord(ch)
    return False if o<0x0250 else any(a<=o<=b for a,b in FOREIGN)
def build_and_locate(conversations, tokenizer, max_len, run=3):
    full="".join(f"<|im_start|>{t['role']}\n{t['content']}<|im_end|>\n" for t in conversations)
    hdr="<|im_start|>assistant\n"; apos=full.rfind(hdr)
    a_char=apos+len(hdr) if apos>=0 else len(full)
    enc=tokenizer(full,return_offsets_mapping=True,truncation=True,max_length=max_len)
    ids=enc["input_ids"]; offs=enc["offset_mapping"]
    A=[i for i,(s,e) in enumerate(offs) if s>=a_char and e>s]
    # char indices of ALL foreign letters (for entry-vs-sustained), + first-run onset
    fchars=[i for i,ch in enumerate(full[a_char:]) if is_foreign(ch)]
    onset=None; cnt=0; start=None
    for i,ch in enumerate(full[a_char:]):
        if is_foreign(ch):
            if cnt==0: start=i
            cnt+=1
            if cnt>=run: onset=start; break
        elif ch.isalpha(): cnt=0; start=None
    def char2tok(c):
        tc=a_char+c
        for i,(s,e) in enumerate(offs):
            if s<=tc<e: return i
        return None
    tip_tok=char2tok(onset) if onset is not None else None
    foreign_toks=sorted({char2tok(c) for c in fchars if char2tok(c) is not None})
    tok2ai={t:i for i,t in enumerate(A)}
    return ids, A, (tok2ai.get(tip_tok) if tip_tok is not None else None), \
           sorted({tok2ai[t] for t in foreign_toks if t in tok2ai})
print("helpers ready")


In [ ]:
# === Cell 6 — CAPTURE pass: save raw routing tensors (resumable) ============
import os, json, hashlib, numpy as np
wp=find_file(DATA_DIR,"weird_transcripts.jsonl"); mp=find_file(DATA_DIR,"weird_meta.jsonl")
assert wp and mp, f"need weird_transcripts + weird_meta under {DATA_DIR}"
trans=read_jsonl(wp); meta=read_jsonl(mp)
by_beh={}
for tr,m in zip(trans,meta):
    by_beh.setdefault(m.get("behavior_id"),[]).append((tr,m))
sel=[]
for b in TIPPED_BEHAVIORS:   sel+=[("tipped",tr,m)  for tr,m in by_beh.get(b,[])[:N_PER_CLASS]]
for b in CONTROL_BEHAVIORS:  sel+=[("control",tr,m) for tr,m in by_beh.get(b,[])[:N_PER_CLASS]]
print({c:sum(1 for x in sel if x[0]==c) for c in ("tipped","control")})

os.makedirs(CAPTURE_DIR,exist_ok=True)
INDEX=os.path.join(CAPTURE_DIR,"index.jsonl")
done={r["id"] for r in (read_jsonl(INDEX) if os.path.exists(INDEX) else [])}
idxf=open(INDEX,"a",encoding="utf-8")
for n,(cls,tr,m) in enumerate(sel,1):
    if tr["id"] in done: continue
    try:
        ids,A,tip_ai,foreign_ai=build_and_locate(tr["conversations"],tokenizer,MAX_LEN,FOREIGN_RUN)
        if len(A)<4: continue
        R=route(ids)
        fn=hashlib.sha1(tr["id"].encode()).hexdigest()[:16]+".npz"
        np.savez_compressed(os.path.join(CAPTURE_DIR,fn),
                            R=R, A=np.array(A,dtype=np.int32))
        idxf.write(json.dumps(dict(id=tr["id"],cls=cls,behavior=m.get("behavior_id"),
                   file=fn,n_assistant=len(A),tip_ai=tip_ai,
                   foreign_ai=foreign_ai[:2000]))+"\n"); idxf.flush()
    except Exception as e:
        print(f"  [{n}] {tr['id'][:24]} error: {type(e).__name__}: {e}")
    if n%20==0: print(f"  {n}/{len(sel)}")
idxf.close(); print("captures ->", CAPTURE_DIR)


## Offline parameter study (no GPU from here on)

Everything below reads only `routing_caps/` from Drive. The core quantity is the
**position-matched shift**: novelty of the tipped transcript at its switch token,
minus the mean control novelty at the *same assistant position*, computed under a
given parameterisation `(W, layers, k)` — with a bootstrap 95% CI over transcripts.


In [ ]:
# === Cell 7 — offline core: load captures + parameterised novelty ===========
import os, json, numpy as np
CAPTURE_DIR = globals().get("CAPTURE_DIR","/content/drive/MyDrive/weirdspec/routing_caps")
if not os.path.exists(CAPTURE_DIR):
    from google.colab import drive; drive.mount("/content/drive")
def read_jsonl(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
index=read_jsonl(os.path.join(CAPTURE_DIR,"index.jsonl"))
# de-duplicate (resume can append twice)
seen=set(); idx=[]
for r in index:
    if r["id"] not in seen: idx.append(r); seen.add(r["id"])
index=idx
print({c:sum(1 for r in index if r["cls"]==c) for c in ("tipped","control")},
      "| with tip:", sum(1 for r in index if r["cls"]=="tipped" and r.get("tip_ai") is not None))

_cache={}
def load(rec):
    if rec["file"] not in _cache:
        z=np.load(os.path.join(CAPTURE_DIR,rec["file"]))
        _cache[rec["file"]]=(z["R"],z["A"])
        if len(_cache)>300: _cache.pop(next(iter(_cache)))
    return _cache[rec["file"]]

def novelty_at(R, A, i, W, layers=None, k=None):
    """Per-layer novelty at assistant index i under (W, layers, k); returns [n_layers_sel]."""
    L=R.shape[0]; k=k or R.shape[2]
    ls=np.arange(L) if layers is None else np.asarray(layers)
    t=int(A[i]); prev=[int(A[j]) for j in range(max(0,i-W),i)]
    if not prev: return None
    S=R[ls,t,:k]                       # [Lsel,k]
    out=np.empty(len(ls))
    for a,l in enumerate(ls):
        past=np.zeros(1024,dtype=bool)
        past[R[l,prev,:k].ravel().astype(np.int32)]=True
        out[a]=1.0-past[S[a].astype(np.int32)].mean()
    return out

def matched_shift(W=16, layers=None, k=None, control_behavior=None, boot=3000, seed=1):
    """Position-matched tip shift under a parameterisation. Returns (mean, lo, hi, n)."""
    ctl=[r for r in index if r["cls"]=="control"
         and (control_behavior is None or r["behavior"]==control_behavior)]
    diffs=[]
    for r in index:
        if r["cls"]!="tipped" or r.get("tip_ai") is None: continue
        p=r["tip_ai"]
        if p<1: continue                     # position 0 has no history at any W
        R,A=load(r)
        nt=novelty_at(R,A,p,W,layers,k)
        if nt is None: continue
        cvals=[]
        for c in ctl:
            if c["n_assistant"]<=p: continue
            Rc,Ac=load(c)
            nc=novelty_at(Rc,Ac,p,W,layers,k)
            if nc is not None: cvals.append(nc.mean())
        if len(cvals)>=10:
            diffs.append(nt.mean()-float(np.mean(cvals)))
    d=np.array(diffs)
    if len(d)<8: return None
    rng=np.random.default_rng(seed)
    bs=[np.mean(rng.choice(d,len(d))) for _ in range(boot)]
    lo,hi=np.percentile(bs,[2.5,97.5])
    return float(d.mean()),float(lo),float(hi),len(d)
print("offline core ready")


In [ ]:
# === Cell 8 — perspective 1+3: window W and top-k depth =====================
import numpy as np, matplotlib.pyplot as plt
Ws=[2,4,8,16,32,64]; Ks=[1,2,4,8]
res_W={w:matched_shift(W=w) for w in Ws}
res_K={kk:matched_shift(W=16,k=kk) for kk in Ks}
print(f"{'W':>4} {'shift':>8} {'95% CI':>20} {'n':>4}")
for w,r in res_W.items():
    if r: print(f"{w:4d} {r[0]:+8.3f} [{r[1]:+.3f},{r[2]:+.3f}] {r[3]:4d}"
                + ("   sig" if r[1]>0 or r[2]<0 else ""))
print(f"\n{'k':>4} {'shift':>8} {'95% CI':>20} {'n':>4}")
for kk,r in res_K.items():
    if r: print(f"{kk:4d} {r[0]:+8.3f} [{r[1]:+.3f},{r[2]:+.3f}] {r[3]:4d}"
                + ("   sig" if r[1]>0 or r[2]<0 else ""))
fig,ax=plt.subplots(1,2,figsize=(11,4))
for a,(xs,res,xl) in zip(ax,[(Ws,res_W,"window W (tokens)"),(Ks,res_K,"top-k depth")]):
    xs2=[x for x in xs if res[x]]; m=[res[x][0] for x in xs2]
    lo=[res[x][1] for x in xs2]; hi=[res[x][2] for x in xs2]
    a.errorbar(xs2,m,yerr=[np.array(m)-lo,np.array(hi)-np.array(m)],fmt="o-",color="#2563EB",capsize=3)
    a.axhline(0,color="black",lw=.8,ls=":"); a.set_xlabel(xl); a.set_ylabel("position-matched shift")
    a.set_xscale("log",base=2); a.grid(True,alpha=.25)
ax[0].set_title("does the break depend on history length?")
ax[1].set_title("dominant expert vs the top-8 tail")
plt.tight_layout(); plt.show()


In [ ]:
# === Cell 9 — perspective 2: layer-resolved — WHERE does the break live? ====
import numpy as np, matplotlib.pyplot as plt
L=load(index[0])[0].shape[0]
lay=[]
for l in range(L):
    r=matched_shift(W=16,layers=[l],boot=800)
    lay.append(r)
m=np.array([r[0] if r else np.nan for r in lay])
lo=np.array([r[1] if r else np.nan for r in lay]); hi=np.array([r[2] if r else np.nan for r in lay])
sig=[l for l in range(L) if lay[l] and lay[l][1]>0]
print("layers with CI>0 (significant positive break):", sig)
print("top-5 layers by shift:", list(np.argsort(-np.nan_to_num(m))[:5]))
fig,axp=plt.subplots(figsize=(10,3.8))
axp.fill_between(range(L),lo,hi,color="#2563EB",alpha=.2,linewidth=0)
axp.plot(range(L),m,color="#2563EB",lw=2)
axp.axhline(0,color="black",lw=.8,ls=":")
axp.set_xlabel("layer"); axp.set_ylabel("position-matched shift")
axp.set_title("layer-resolved routing break at the switch (mean, 95% CI)")
axp.grid(True,alpha=.25); plt.tight_layout(); plt.show()


In [ ]:
# === Cell 10 — perspective 4+5: control choice + entry vs sustained =========
import numpy as np
print("== control-behaviour dependence (W=16, all layers, k=8) ==")
behs=sorted({r["behavior"] for r in index if r["cls"]=="control"})
for b in behs+[None]:
    r=matched_shift(W=16,control_behavior=b)
    if r: print(f"  vs {str(b or 'ALL CONTROLS'):32s} {r[0]:+.3f} [{r[1]:+.3f},{r[2]:+.3f}] n={r[3]}"
                + ("   sig" if r[1]>0 or r[2]<0 else ""))

print("\n== entry vs sustained foreign text (script confound check) ==")
# entry: the switch token itself.  sustained: foreign tokens >=20 positions past the onset.
ctl=[r for r in index if r["cls"]=="control"]
ent,sus=[],[]
for r in index:
    if r["cls"]!="tipped" or r.get("tip_ai") is None: continue
    R,A=load(r); p=r["tip_ai"]
    later=[i for i in r.get("foreign_ai",[]) if i>=p+20 and i<r["n_assistant"]]
    def ctl_mean_at(q):
        vals=[]
        for c in ctl:
            if c["n_assistant"]<=q: continue
            nc=novelty_at(*load(c),q,16)
            if nc is not None: vals.append(nc.mean())
        return float(np.mean(vals)) if len(vals)>=10 else None
    if p>=1:
        nt=novelty_at(R,A,p,16); cm=ctl_mean_at(p)
        if nt is not None and cm is not None: ent.append(nt.mean()-cm)
    if later:
        q=later[len(later)//2]
        nq=novelty_at(R,A,q,16); cm=ctl_mean_at(q)
        if nq is not None and cm is not None: sus.append(nq.mean()-cm)
rng=np.random.default_rng(3)
for name,dd in (("entry (switch token)",ent),("sustained foreign (>=20 later)",sus)):
    d=np.array(dd)
    if len(d)>=8:
        bs=[np.mean(rng.choice(d,len(d))) for _ in range(3000)]
        lo,hi=np.percentile(bs,[2.5,97.5])
        print(f"  {name:32s} {d.mean():+.3f} [{lo:+.3f},{hi:+.3f}] n={len(d)}"
              + ("   sig" if lo>0 or hi<0 else ""))
    else:
        print(f"  {name:32s} n={len(d)} — too few")
print("\nreading: entry sig + sustained ~0  => a genuine mode-ENTRY event, not a script artifact.")
print("         both sig                  => foreign-script tokens route differently throughout")
print("                                      (the confound matters; entry minus sustained is the real entry effect).")


In [ ]:
# === Cell 11 — ZOOM: the reshuffling itself, in three resolutions ===========
# (offline; needs only routing_caps/. Standalone: defines its own helpers.)
# Liest nur routing_caps/ von Drive. In frischer Session: erst Cell 7 (offline core),
# dann diese Zelle — oder standalone: sie definiert alles Noetige selbst.
import os, json
import numpy as np, matplotlib.pyplot as plt
CAPTURE_DIR = globals().get("CAPTURE_DIR","/content/drive/MyDrive/weirdspec/routing_caps")
if not os.path.exists(CAPTURE_DIR):
    from google.colab import drive; drive.mount("/content/drive")
def _readjs(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
index=_readjs(os.path.join(CAPTURE_DIR,"index.jsonl"))
seen=set(); index=[r for r in index if not (r["id"] in seen or seen.add(r["id"]))]
_zc={}
def zload(rec):
    if rec["file"] not in _zc:
        z=np.load(os.path.join(CAPTURE_DIR,rec["file"]))
        _zc[rec["file"]]=(z["R"],z["A"])
        if len(_zc)>300: _zc.pop(next(iter(_zc)))
    return _zc[rec["file"]]
W=16
def lay_novelty(R,A,i,k=None):
    """Novelty pro Layer an Assistant-Index i (Fenster W). None wenn i<1."""
    if i<1 or i>=len(A): return None
    k=k or R.shape[2]; L=R.shape[0]
    prev=[int(A[j]) for j in range(max(0,i-W),i)]
    out=np.empty(L)
    for l in range(L):
        past=np.zeros(1024,bool); past[R[l,prev,:k].ravel().astype(np.int32)]=True
        out[l]=1.0-past[R[l,int(A[i]),:k].astype(np.int32)].mean()
    return out
def new_by_rank(R,A,i):
    """Bool [L,K]: ist der Experte auf Rang r an Position i neu vs. Fenster?"""
    if i<1 or i>=len(A): return None
    L,K=R.shape[0],R.shape[2]
    prev=[int(A[j]) for j in range(max(0,i-W),i)]
    out=np.zeros((L,K),bool)
    for l in range(L):
        past=np.zeros(1024,bool); past[R[l,prev,:].ravel().astype(np.int32)]=True
        out[l]=~past[R[l,int(A[i]),:].astype(np.int32)]
    return out

tipped=[r for r in index if r["cls"]=="tipped" and r.get("tip_ai") is not None and r["tip_ai"]>=1]
ctl=[r for r in index if r["cls"]=="control"]
print(f"{len(tipped)} tipped mit Tip, {len(ctl)} Kontrollen")
DTS=list(range(-4,13))

# ---- (C) Layer x Token: positions-gematchte Excess-Novelty um den Switch ----
_ctl_cache={}
def ctl_layer_mean(p):
    if p not in _ctl_cache:
        vals=[]
        for c in ctl:
            if c["n_assistant"]<=p: continue
            nv=lay_novelty(*zload(c),p)
            if nv is not None: vals.append(nv)
        _ctl_cache[p]=np.mean(vals,axis=0) if len(vals)>=10 else None
    return _ctl_cache[p]
L=zload(index[0])[0].shape[0]
H=np.full((L,len(DTS)),np.nan); Hn=np.zeros(len(DTS),int)
acc=[[[] for _ in DTS] for _ in range(L)]
for r in tipped:
    R,A=zload(r); p0=r["tip_ai"]
    for di,dt in enumerate(DTS):
        i=p0+dt
        nv=lay_novelty(R,A,i)
        cm=ctl_layer_mean(i) if nv is not None else None
        if nv is not None and cm is not None:
            for l in range(L): acc[l][di].append(nv[l]-cm[l])
for l in range(L):
    for di in range(len(DTS)):
        if len(acc[l][di])>=8: H[l,di]=np.mean(acc[l][di])
Hn=[len(acc[0][di]) for di in range(len(DTS))]

# ---- (B) Rang x Token: wer wird ausgetauscht — der Anfuehrer oder der Tail? ----
K=zload(index[0])[0].shape[2]
def rank_map(rows, tip_of):
    M=np.full((K,len(DTS)),np.nan)
    cnt=[[[] for _ in DTS] for _ in range(K)]
    for r in rows:
        R,A=zload(r); p0=tip_of(r)
        if p0 is None: continue
        for di,dt in enumerate(DTS):
            nb=new_by_rank(R,A,p0+dt)
            if nb is not None:
                for k_ in range(K): cnt[k_][di].append(nb[:,k_].mean())
    for k_ in range(K):
        for di in range(len(DTS)):
            if len(cnt[k_][di])>=8: M[k_,di]=np.mean(cnt[k_][di])
    return M
rng=np.random.default_rng(5)
tip_positions=[r["tip_ai"] for r in tipped]
MB_t=rank_map(tipped, lambda r: r["tip_ai"])
MB_c=rank_map(ctl, lambda r: int(rng.choice(tip_positions)) if r["n_assistant"]>max(tip_positions) else None)

# ---- (A) Mikroskop: ein Transkript, staerkster Layer, Experten-Raster ----
best=None; best_val=-9
for r in tipped:
    if r["tip_ai"]<4: continue
    nv=lay_novelty(*zload(r), r["tip_ai"])
    if nv is not None and nv.mean()>best_val: best_val=nv.mean(); best=r
if best is None: best=max(tipped,key=lambda r:r["tip_ai"])
Rb,Ab=zload(best); p0=best["tip_ai"]
nv=lay_novelty(Rb,Ab,p0); lstar=int(np.argmax(nv)) if nv is not None else 0
lo_,hi_=max(0,p0-8), min(best["n_assistant"],p0+17)
toks=list(range(lo_,hi_))
experts=[]
for i in toks:
    for e in Rb[lstar,int(Ab[i]),:]:
        if int(e) not in experts: experts.append(int(e))
E=len(experts); G=np.zeros((E,len(toks))); R1=np.zeros((E,len(toks)),bool)
eidx={e:j for j,e in enumerate(experts)}
for c_,i in enumerate(toks):
    sel=Rb[lstar,int(Ab[i]),:]
    for rk,e in enumerate(sel):
        G[eidx[int(e)],c_]=1; R1[eidx[int(e)],c_] |= (rk==0)

# ---------------------------------- Plots ----------------------------------
fig,ax=plt.subplots(1,3,figsize=(16.5,4.6))
# (A) Mikroskop-Raster
ax[0].imshow(G,aspect="auto",cmap="Blues",vmin=0,vmax=1.4,interpolation="nearest")
ys,xs=np.where(R1); ax[0].scatter(xs,ys,s=12,color="#111827",marker="s",label="Rang-1-Experte")
ax[0].axvline(p0-lo_-0.5,color="#DC2626",lw=1.5)
ax[0].set_xticks(range(0,len(toks),4)); ax[0].set_xticklabels([t-p0 for t in toks][::4])
ax[0].set_xlabel("Token relativ zum Switch"); ax[0].set_ylabel(f"Experten-ID (Layer {lstar})")
ax[0].set_yticks(range(E)); ax[0].set_yticklabels(experts,fontsize=6)
ax[0].set_title(f"Mikroskop: ein Transkript, Layer {lstar}\n(blau = in top-8; ■ = Rang 1)")
ax[0].legend(loc="upper left",fontsize=8,frameon=False)
# (B) Rang x Token, tipped vs control
vmax=np.nanmax([np.nanmax(MB_t),np.nanmax(MB_c)])
im=ax[1].imshow(MB_t,aspect="auto",cmap="viridis",vmin=0,vmax=vmax,interpolation="nearest")
ax[1].set_yticks(range(K)); ax[1].set_yticklabels([f"Rang {k_+1}" for k_ in range(K)],fontsize=8)
ax[1].set_xticks(range(0,len(DTS),2)); ax[1].set_xticklabels(DTS[::2])
ax[1].axvline(DTS.index(0)-0.5,color="#DC2626",lw=1.5)
ax[1].set_xlabel("Token relativ zum Switch")
ax[1].set_title("Neu-Anteil je Rang — TIPPED\n(Rang 1 bleibt, der Tail wird umgesteckt?)")
plt.colorbar(im,ax=ax[1],fraction=0.045)
# (C) Layer x Token Excess
vm=np.nanmax(np.abs(H))
im2=ax[2].imshow(H,aspect="auto",cmap="RdBu_r",vmin=-vm,vmax=vm,interpolation="nearest")
ax[2].axvline(DTS.index(0)-0.5,color="#111827",lw=1.2,ls=":")
ax[2].set_xticks(range(0,len(DTS),2)); ax[2].set_xticklabels(DTS[::2])
ax[2].set_xlabel("Token relativ zum Switch"); ax[2].set_ylabel("Layer")
ax[2].set_title("Excess-Novelty vs Kontrollen\n(Layer x Token, positions-gematcht)")
plt.colorbar(im2,ax=ax[2],fraction=0.045)
plt.tight_layout(); plt.show()

fig2,axc=plt.subplots(figsize=(7,3.6))
imc=axc.imshow(MB_c,aspect="auto",cmap="viridis",vmin=0,vmax=vmax,interpolation="nearest")
axc.set_yticks(range(K)); axc.set_yticklabels([f"Rang {k_+1}" for k_ in range(K)],fontsize=8)
axc.set_xticks(range(0,len(DTS),2)); axc.set_xticklabels(DTS[::2])
axc.axvline(DTS.index(0)-0.5,color="#DC2626",lw=1.5)
axc.set_xlabel("Token relativ zum Pseudo-Switch"); axc.set_title("Neu-Anteil je Rang — KONTROLLE (Pseudo-Tip)")
plt.colorbar(imc,ax=axc,fraction=0.045)
plt.tight_layout(); plt.show()
print(f"Spalten-n (tipped): {Hn}")
print(f"Beispiel-Transkript: {best['id'][:40]}  tip@{best['tip_ai']}  staerkster Layer: {lstar}")


### Reading the study

* **W sweep** — if the shift is flat in `W`, the break is *local* (one token's experts
  vs its immediate past). Growth with `W` means the switch keeps recruiting experts
  that were not used anywhere in a long history — a deeper regime change.
* **Layer profile** — a break concentrated in early layers points at lexical/script
  routing; mid/late layers point at a semantic mode change. The significant-layer
  list is the fingerprint's *address*.
* **Top-k depth** — significant at `k=1` means the *dominant* expert flips at the
  switch; only at `k=8` means the tail reshuffles while the leader stays.
* **Control choice** — the effect should hold against *both* controls; if it holds
  against one only, the "discontinuity" partly reflects that control's own quirks.
* **Entry vs sustained** — the cleanest interpretation gate: entry-only ⇒ a
  mode-entry event; both ⇒ script-driven routing throughout, and the entry-minus-
  sustained difference is the honest entry effect.
